In [16]:
# The game must use a 6x6 board that is turned diagonally (see illustration).

import heapq
import random
import copy

In [17]:
# initialize board 2.0 (team, are you okay with this update?)
def initialize_board(size=6):
    board = []
    # top half
    for i in range(1, size + 1):
        spaces = size - i
        row = [' ' for _ in range(spaces)]

        if i == 1:
            row.append('f')   # fox at top
        else:
            color = 'b' if i % 2 == 1 else 'w'
            row.extend([color] * i)

        board.append(row)

    # bottom half with same logic
    for i in range(size - 1, 0, -1):
        spaces = size - i
        row = [' ' for _ in range(spaces)]

        if i == 3:
            # place hounds on alternating dark squares
            # row.extend(['b', 'h', 'b', 'h', 'b'])  # ensures hounds are on or next to 'b'
            row.extend('h' * i)
        elif i == 1:
            # row.append('b')  # dark square for final hound
            row.append('h')  # final hound / goal spot
        else:
            color = 'b' if i % 2 == 1 else 'w'
            row.extend([color] * i)

        board.append(row)

    # extra padding row for alignment (same length as top padding)
    board.append([' ' for _ in range(size)])
    return board

In [18]:
def print_board(board):
    for row in board:
        # count leading spaces for diamond shape
        leading = 0
        while leading < len(row) and row[leading] == ' ':
            leading += 1
        line = ' ' * leading + ' '.join(row[leading:])
        print(line)
    print()


In [19]:
def valid(board, r, c):
    return 0 <= r < len(board) and 0 <= c < len(board[r])

In [20]:
def apply_move(board, old_pos, new_pos, piece):
    new_board = [list(row) for row in board]
    old_r, old_c = old_pos
    new_r, new_c = new_pos
    new_board[old_r][old_c] = 'b'
    new_board[new_r][new_c] = piece
    return ["".join(row) for row in new_board]

In [31]:
def Fox_Short_Path_AI(board):
    """
    Returns: (from_pos, to_pos, True, message, goal_reached)
    Does NOT modify `board` (caller should apply the move).
    Uses Dijkstra (priority queue) with a small penalty for moving next to hounds.
    Move rules match Fox_Random_AI: orthogonal neighbor 'b' OR jump over 'w' into 'b'.
    """

    def valid(r, c):
        return 0 <= r < len(board) and 0 <= c < len(board[r])

    def legal_targets_from(r, c):
        # same rules as your Fox_Random_AI: orthogonal moves; if adjacent is 'w' can jump to next 'b'
        res = []
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if not valid(nr, nc):
                continue
            val = board[nr][nc]
            if val == 'b':
                res.append((nr, nc))
            elif val == 'w':  # can jump over a white square to a black square
                jr, jc = nr + dr, nc + dc
                if valid(jr, jc) and board[jr][jc] == 'b':
                    res.append((jr, jc))
        return res

    def hound_distance_penalty(r, c):
        # small penalty if a hound sits on an orthogonal neighbor (tweakable)
        penalty = 0
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if valid(nr, nc) and board[nr][nc] == 'h':
                penalty += 2
        return penalty

    # find fox position
    fox_pos = None
    for r in range(len(board)):
      for c in range(len(board[r])):
        if board[r][c] == 'f':
            fox_pos = (r, c)
            break
      if fox_pos:
        break

    if not fox_pos:
      return None, None, "no fox found", False, False

    sr, sc = fox_pos

# Fox always starts at top and must reach bottom
    goal_row = len(board) - 2

    # Dijkstra using (cost, (r,c), path_list)
    pq = [(0, (sr, sc), [])]
    visited = set()

    while pq:
        cost, (r, c), path = heapq.heappop(pq)
        if (r, c) in visited:
            continue
        visited.add((r, c))

        if r == goal_row:
            full_path = path + [(r, c)]   # includes start as first item (if path not empty)
            # Sanity: full_path[0] should be the start
            # If full_path length >= 2, we can take next step
            if len(full_path) >= 2:
                nr, nc = full_path[1]
                goal_reached = (nr == goal_row)
                return (sr, sc), (nr, nc), True, f"moved from {(sr,sc)} to {(nr,nc)}", goal_reached
            else:
                # already at goal (rare), return stay
                return (sr, sc), (sr, sc), True, "already at goal", True

        for nr, nc in legal_targets_from(r, c):
            if (nr, nc) in visited:
                continue
            move_cost = 1 + hound_distance_penalty(nr, nc)
            heapq.heappush(pq, (cost + move_cost, (nr, nc), path + [(r, c)]))

    # no path found -> signal and let caller handle (you can fall back to random AI)
    return None, None, "no path found", False, False

In [32]:
'''
def main():
  board = initialize_board(6)
  print("Step 0: initial board")
  print_board(board)

  for step in range(1, 11):  # simulate 10 moves
      old_pos, new_pos, moved, msg, goal = Fox_Short_Path_AI(board)
      print(f"Step {step}: {msg}")
      board = apply_move(board, old_pos, new_pos, 'f')
      print_board(board)
      if not moved:
          print("Fox is trapped! Game over.")
          break
      if goal:
          print("Fox reached the goal! Fox wins!")
          break

# run the main function
main()
'''


def initialize_temp_board(size=6):
    '''
    Test simulation for Fox_Short_Path_AI using a temporary board with no hounds.
    Returns: None
    Modifies only a temporary test board (does NOT affect the main project board).
    Prints the board and AI moves step by step for up to 10 moves.
    Uses the same move rules as Fox_Short_Path_AI
    Designed to verify that the Fox AI can reach the goal without interference from hounds.
    '''

    board = []
    # Top half
    for i in range(1, size + 1):
        spaces = size - i
        row = [' ' for _ in range(spaces)]
        if i == 1:
            row.append('f')  # Fox at top
        else:
            color = 'w' if i % 2 == 0 else 'b'  # Even i: 'w', Odd i: 'b'
            row.extend([color] * i)
        board.append(row)

    # Bottom half (no hounds)
    for i in range(size - 1, 0, -1):
        spaces = size - i
        row = [' ' for _ in range(spaces)]
        if i == 3:
            row.extend(['b'] * i)  # No hounds, all 'b'
        elif i == 1:
            row.append('b')  # Goal spot
        else:
            color = 'w' if i % 2 == 0 else 'b'
            row.extend([color] * i)
        board.append(row)

    # Extra padding row
    board.append([' ' for _ in range(size)])
    return board

# Modified main function with debugging for no-hounds test
def main_nohounds():
    board = initialize_temp_board(6)
    print("Step 0: initial board")
    print_board(board)

    # Define legal_targets_from for debugging (copied from Fox_Short_Path_AI)
    def valid(r, c):
        return 0 <= r < len(board) and 0 <= c < len(board[r])

    def legal_targets_from(r, c):
        res = []
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if not valid(nr, nc):
                continue
            val = board[nr][nc]
            if val == 'b':
                res.append((nr, nc))
            elif val == 'w':
                jr, jc = nr + dr, nc + dc
                if valid(jr, jc) and board[jr][jc] == 'b':
                    res.append((jr, jc))
        return res

    for step in range(1, 11):  # Simulate up to 10 moves
        # Find fox position for debugging
        fox_pos = None
        for r in range(len(board)):
            for c in range(len(board[r])):
                if board[r][c] == 'f':
                    fox_pos = (r, c)
                    break
            if fox_pos:
                break

        if fox_pos:
            print(f"Fox at {fox_pos}, legal moves: {legal_targets_from(*fox_pos)}")

        old_pos, new_pos, moved, msg, goal = Fox_Short_Path_AI(board)
        print(f"Step {step}: {msg}")
        if not moved:
            print("Fox is trapped! Game over.")
            break
        if old_pos is None or new_pos is None:
            print("Invalid move returned. Check Fox_Short_Path_AI.")
            break
        board = apply_move(board, old_pos, new_pos, 'f')
        print_board(board)
        if goal:
            print("Fox reached the goal! Fox wins!")
            break

# Run the test
main_nohounds()



Step 0: initial board
     f
    w w
   b b b
  w w w w
 b b b b b
w w w w w w
 b b b b b
  w w w w
   b b b
    w w
     b
      

Fox at (0, 5), legal moves: [(2, 5)]
Step 1: moved from (0, 5) to (2, 5)
     b
    w w
   b b f
  w w w w
 b b b b b
w w w w w w
 b b b b b
  w w w w
   b b b
    w w
     b
      

Fox at (2, 5), legal moves: [(0, 5), (4, 5), (2, 4)]
Step 2: moved from (2, 5) to (4, 5)
     b
    w w
   b b b
  w w w w
 b b b b f
w w w w w w
 b b b b b
  w w w w
   b b b
    w w
     b
      

Fox at (4, 5), legal moves: [(2, 5), (6, 5), (4, 4)]
Step 3: moved from (4, 5) to (6, 5)
     b
    w w
   b b b
  w w w w
 b b b b b
w w w w w w
 b b b b f
  w w w w
   b b b
    w w
     b
      

Fox at (6, 5), legal moves: [(4, 5), (8, 5), (6, 4)]
Step 4: moved from (6, 5) to (8, 5)
     b
    w w
   b b b
  w w w w
 b b b b b
w w w w w w
 b b b b b
  w w w w
   b b f
    w w
     b
      

Fox at (8, 5), legal moves: [(6, 5), (10, 5), (8, 4)]
Step 5: moved from (8, 5) to (10, 

In [23]:
def Hounds_Minimax_AI(board, depth, alpha, beta, isMax):
  def find_fox(board):
    for r in range(len(board)):
      for c in range(len(board[r])):
        if board[r][c] == 'f':
          return (r, c)
    return None

  def fox_moves(board, r, c):
      directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
      moves = []
      for dr, dc in directions:
        nr, nc = r + dr, c + dc
        if valid(board, nr, nc):
          if board[nr][nc] == 'b':
            moves.append((nr, nc))
          elif board[nr][nc] == 'w':
            jr, jc = nr + dr, nc + dc
            if valid(board, jr, jc) and board[jr][jc] == 'b':
              moves.append((jr, jc))
      return moves

  def get_hounds(board):
    return [(r, c) for r in range(len(board)) for c in range(len(board[r])) if board[r][c] == 'h']

  def get_hound_moves(board, r, c):
      directions = [(-1, 0), (0, -1), (0, 1)]  # up, left, right
      moves = []
      for dr, dc in directions:
        nr, nc = r + dr, c + dc
        if valid(board, nr, nc):
          if board[nr][nc] == 'b':
            moves.append((nr, nc))
          elif board[nr][nc] == 'w':
            jr, jc = nr + dr, nc + dc
            if valid(board,jr, jc) and board[jr][jc] == 'b':
              moves.append((jr, jc))
      return moves

  def apply_new_state(board, old_r, old_c, new_r, new_c, piece):
    new_board = [list(row) for row in board]
    new_board[old_r][old_c] = 'b'    # old spot now blank
    new_board[new_r][new_c] = piece  # new spot with hound
    return ["".join(row) for row in new_board]

  def evaluate(board):
      fox_pos = find_fox(board)
      if not fox_pos:
          return float('inf')  # no fox present
      fox_r, fox_c = fox_pos
      if fox_r == 10:
        return float('-inf')  # fox reached the goal
      moves = fox_moves(board, fox_r, fox_c)
      if len(moves) == 0:
        return float('inf')  # fox is trapped

      return -len(moves) + (len(board) - fox_r) * 0.5

#   if depth == 0:
#     return evaluate(board), None, None

#   # Base case for finding fox
#   fox_pos = find_fox(board)
#   if fox_pos is None:
#     return float('inf'), None, True

#   if fox_moves(board, *fox_pos) == 0:
#     return float('inf'), None, True

  if depth == 0:
    # return evaluate(board), None, (len(fox_moves(board, *fox_pos)) == 0)
    return evaluate(board), None

  if isMax:
    best = float('-inf')
    best_move = None
    hounds = get_hounds(board)
    # hounds_moved = False
    for (r, c) in hounds:
      for (nr, nc) in get_hound_moves(board, r, c):
        # hounds_moved = True
        new_board = apply_new_state(board, r, c, nr, nc, 'h')
        val, _ = Hounds_Minimax_AI(new_board, depth-1, alpha, beta, False)
        if val is None: #make sure val is always a number
          val = float('-inf')
        if val > best:
          best = val
          best_move = ((r, c), (nr, nc))
        alpha = max(alpha, val)
        if beta <= alpha:
            break
      if beta <= alpha:
          break
    # if not hounds_moved:
    #   return evaluate(board), None, (len(fox_moves(board, *fox_pos)) == 0 if fox_pos else True)

    # fox_pos = find_fox(board)
    # trapped = (fox_moves(board, *fox_pos) == 0) if fox_pos else True
    # return best, best_move, trapped
    return best, best_move

  else:
    best = float('inf')
    best_move = None
    fox_pos = find_fox(board)
    if not fox_pos:
        return 1000, None

    fox_r, fox_c = fox_pos
    if not fox_moves(board, fox_r, fox_c):
      return float('inf'), None
    for (nr, nc) in fox_moves(board, fox_r, fox_c):
        new_board = apply_new_state(board, fox_r, fox_c, nr, nc, 'f')
        val, _ = Hounds_Minimax_AI(new_board, depth-1, alpha, beta, True)
        # best = min(best, val)
        if val < best:
          best = val
          best_move = ((fox_r, fox_c), (nr, nc))
        beta = min(beta, val)
        if beta <= alpha:
          break

    # fox_pos = find_fox(board)
    # trapped = len(fox_moves(*fox_pos)) == 1 if fox_pos else True
    # trapped = (fox_moves(board, *fox_pos) == 0)
    # print(trapped)

    # return best, best_move, trapped
    return best, best_move

In [33]:
def play_game(max_turns=100):
    board = initialize_board()
    print("Initial Board:")
    print_board(board)

    turn = 0
    fox_turn = True

    while turn < max_turns:
        print(f"--- Turn {turn + 1} ({'Fox' if fox_turn else 'Hounds'}) ---")

        if fox_turn:
            # FOX AI
            old_pos, new_pos, ok, msg, goal_reached = Fox_Short_Path_AI(board)
            if not ok or not old_pos or not new_pos:
                print("Fox has no valid moves! Hounds win!")
                break

            print(msg)
            board = apply_move(board, old_pos, new_pos, 'f')
            print_board(board)

            if goal_reached:
                print("🦊 Fox reached the goal! Fox wins!")
                break
        else:
            # HOUNDS AI
            val, move = Hounds_Minimax_AI(board, depth=2)
            if move is None:
                print("Hounds have no valid moves! Fox wins!")
                break

            (r1, c1), (r2, c2) = move
            board = apply_move(board, (r1, c1), (r2, c2), 'h')
            print(f"Hounds moved {(r1, c1)} → {(r2, c2)} (eval={val:.2f})")
            print_board(board)

            # Check if fox is trapped
            fox_pos = [(r, c) for r in range(len(board)) for c in range(len(board[r])) if board[r][c] == 'f']
            if not fox_pos:
                print("🐕 Hounds caught the fox! Hounds win!")
                break

        fox_turn = not fox_turn
        turn += 1

    if turn >= max_turns:
        print("Game ended in a draw (turn limit reached).")

# --- Run it ---
play_game()

Initial Board:
     f
    w w
   b b b
  w w w w
 b b b b b
w w w w w w
 b b b b b
  w w w w
   h h h
    w w
     h
      

--- Turn 1 (Fox) ---
Fox has no valid moves! Hounds win!


In [ ]:
# Fox Random AI vs. Hounds Minimax AI
def main():
    board = initialize_board(6)
    print("Step 0: initial board")
    print_board(board)
    def valid(board, r, c):
      return 0 <= r < len(board) and 0 <= c < len(board[r])

    fox_points = 0
    hound_points = 0
    for game in range(1, 101): # play 100 games
      for step in range(1, 101):  # simulate 10 moves
          old_pos, new_pos, ok, moved, msg = Fox_Short_Path_AI(board)
          if not moved:
              print("Fox never moved! Hounds win!")
              hound_points += 1
              break
          if old_pos is None or new_pos is None:
              print("Invalid move returned. Check Fox_Short_Path_AI.")
              break
          print(f"Step {step} (Fox): {msg}")
          if step % 5 == 0:
            if valid(board, old_pos[0]+2, old_pos[1]+1) and board[old_pos[0]+2][old_pos[1]] == 'b':
                print("Fox using force move")
                new_pos = (old_pos[0] + 2, old_pos[1])
            elif valid(board, old_pos[0]+2, old_pos[1]-1) and board[old_pos[0]+2][old_pos[1]-1] == 'b':
                print("Fox using force move")
                new_pos = (old_pos[0] + 2, old_pos[1]-1)
            elif valid(board, old_pos[0]+2, old_pos[1]+1) and board[old_pos[0]+2][old_pos[1]+1] == 'b':
                new_pos = (old_pos[0] + 2, old_pos[1]+1)
                print("Fox using force move")

          board = apply_move(board, old_pos, new_pos, 'f')
          print_board(board)
          goal_reached = (new_pos[0] == len(board) - 2 and board[new_pos[0]][new_pos[1]] == 'f')
          if goal_reached:
              print("Fox reached the goal! Fox wins!")
              fox_points += 1
              break

          best_value, best_move = Hounds_Minimax_AI(board, 5, -float('inf'), float('inf'), True)
          if best_move:
            old_pos, new_pos = best_move
            print(f"\nStep {step} (Hounds): moved from {old_pos} to {new_pos}")
            board = apply_move(board, old_pos, new_pos, 'h')
            print_board(board)

    print(f"Fox points: {fox_points}, Hound points: {hound_points}")

main()

Step 0: initial board
     f
    w w
   b b b
  w w w w
 b b b b b
w w w w w w
 b b b b b
  w w w w
   h h h
    w w
     h
      

Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Invalid move returned. Check Fox_Short_Path_AI.
Inva